# The Pensieve Project — Frontend no Colab

Este notebook sobe uma interface web simples na porta **3000** usando o próprio runtime do Colab.

- **Forensic Core**: timeline, correlação, SQLite, dashboard, sem IA.
- **AI Analyst**: GLiNER + Qwen ligados por padrão.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO = 'https://github.com/Ridd1kulusC0d3r/ThePensieveProject.git'
REF = 'main'
ROOT = Path('/content/ThePensieveProject')

if not ROOT.exists():
    subprocess.run(['git','clone',REPO,str(ROOT)], check=True)
subprocess.run(['git','-C',str(ROOT),'fetch','--all','--prune'], check=True)
subprocess.run(['git','-C',str(ROOT),'checkout',REF], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[ui,osint,llm]'], check=True)
print('Pensieve instalado.')


## Opcional: pré-carregar GLiNER + Qwen

Deixe `PREFETCH_MODELS = True` para que a primeira investigação AI não precise baixar os modelos no meio da execução.


In [ ]:
PREFETCH_MODELS = True

if PREFETCH_MODELS:
    from huggingface_hub import snapshot_download
    for model in ['urchade/gliner_multi-v2.1','Qwen/Qwen3-0.6B']:
        print('Baixando:', model)
        snapshot_download(repo_id=model)
    print('Modelos prontos.')


## Subir o frontend

O servidor roda em background na porta 3000. Reexecutar esta célula encerra o processo anterior iniciado pelo notebook.


In [ ]:
import os, signal

PID_FILE = Path('/content/pensieve-web.pid')
LOG_FILE = Path('/content/pensieve-web.log')

if PID_FILE.exists():
    try:
        old_pid = int(PID_FILE.read_text().strip())
        os.kill(old_pid, signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass

log = LOG_FILE.open('w')
proc = subprocess.Popen([
    'pensieve-web',
    '--host','0.0.0.0',
    '--port','3000',
    '--case-root','/content/pensieve-cases',
    '--upload-root','/content/pensieve-uploads',
], stdout=log, stderr=subprocess.STDOUT)
PID_FILE.write_text(str(proc.pid))
time.sleep(2)
print('PID:', proc.pid)
print('Log:', LOG_FILE)


## Abrir a interface

O Colab cria uma URL `*.prod.colab.dev` para a porta 3000. É essa URL que você pode manter aberta em outra aba.


In [ ]:
from google.colab import output

try:
    output.serve_kernel_port_as_window(3000)
except AttributeError:
    output.serve_kernel_port_as_iframe(3000, height=900)


## Uso

1. Envie um CSV/TSV/JSONL no painel.
2. Escolha `Forensic Core` ou `AI Analyst`.
3. Em AI, GLiNER e Qwen já começam marcados.
4. Execute a investigação.
5. Navegue em `Resumo`, `Timeline`, `Entidades`, `AI` e `Arquivos`.

Casos ficam em `/content/pensieve-cases/` até o runtime do Colab ser encerrado.
